In [2]:
from google_play_scraper import Sort, reviews
import pandas as pd
from datetime import datetime
import os

def rescrape_im3_with_version():
    APP_ID = 'com.pure.indosat.care'
    T0_BOUNDARY = '2026-04-20'
    OUTPUT_FILE = 'dataset_indosat/myim3_reviews_raw.csv' # Menimpa file lama
    
    print(f"[*] Menginisialisasi ekstraksi ulang untuk: {APP_ID}")
    target_date = datetime.strptime(T0_BOUNDARY, '%Y-%m-%d')
    
    all_reviews = []
    continuation_token = None
    reached_boundary = False
    
    while not reached_boundary:
        result, continuation_token = reviews(
            APP_ID,
            lang='id', 
            country='id', 
            sort=Sort.NEWEST,
            count=1000,
            continuation_token=continuation_token
        )
        
        if not result:
            print("[!] API Google Play berhenti mengembalikan data.")
            break
            
        oldest_in_batch = result[-1]['at']
        all_reviews.extend(result)
        
        print(f" -> Menarik {len(all_reviews)} ulasan... Waktu ulasan tertua di batch: {oldest_in_batch.date()}")
        
        if oldest_in_batch < target_date:
            reached_boundary = True
            print("[*] Batas T0 tercapai.")
            
        if not continuation_token:
            break

    # Konversi ke DataFrame
    df = pd.DataFrame(all_reviews)
    
    # [KUNCI ARSITEKTUR] Menyertakan reviewCreatedVersion secara eksplisit
    target_columns = ['reviewId', 'userName', 'content', 'score', 'at', 'replyContent', 'reviewCreatedVersion']
    
    # Filter hanya kolom yang eksis dari API untuk menghindari KeyError
    available_columns = [col for col in target_columns if col in df.columns]
    df_filtered = df[available_columns].copy()
    
    # Pemangkasan presisi berdasarkan T0
    df_filtered['at'] = pd.to_datetime(df_filtered['at'])
    df_final = df_filtered[df_filtered['at'] >= target_date]
    
    # Ekspor dan timpa CSV mentah sebelumnya
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    df_final.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\n[SUCCESS] Pipeline Ingestion Diperbarui.")
    print(f" -> Total Data Tersimpan: {len(df_final)} baris.")
    print(f" -> Kolom yang diekstrak: {list(df_final.columns)}")
    print(f" -> Data diekspor ke: {OUTPUT_FILE}")

# Eksekusi fungsi di cell notebook
rescrape_im3_with_version()

[*] Menginisialisasi ekstraksi ulang untuk: com.pure.indosat.care
 -> Menarik 1000 ulasan... Waktu ulasan tertua di batch: 2026-06-24
 -> Menarik 2000 ulasan... Waktu ulasan tertua di batch: 2026-06-20
 -> Menarik 3000 ulasan... Waktu ulasan tertua di batch: 2026-06-16
 -> Menarik 4000 ulasan... Waktu ulasan tertua di batch: 2026-06-11
 -> Menarik 5000 ulasan... Waktu ulasan tertua di batch: 2026-06-07
 -> Menarik 6000 ulasan... Waktu ulasan tertua di batch: 2026-06-03
 -> Menarik 7000 ulasan... Waktu ulasan tertua di batch: 2026-05-22
 -> Menarik 8000 ulasan... Waktu ulasan tertua di batch: 2026-05-15
 -> Menarik 9000 ulasan... Waktu ulasan tertua di batch: 2026-05-10
 -> Menarik 10000 ulasan... Waktu ulasan tertua di batch: 2026-05-03
 -> Menarik 11000 ulasan... Waktu ulasan tertua di batch: 2026-04-23
 -> Menarik 12000 ulasan... Waktu ulasan tertua di batch: 2026-04-17
[*] Batas T0 tercapai.

[SUCCESS] Pipeline Ingestion Diperbarui.
 -> Total Data Tersimpan: 11566 baris.
 -> Kolom y

In [3]:
import pandas as pd

def check_time_boundary(csv_file):
    print(f"[*] Menginspeksi rentang waktu: {csv_file}")
    try:
        # Kita baca file mentah untuk kecepatan, hanya mengambil kolom 'at'
        df = pd.read_csv(csv_file, usecols=['at'], parse_dates=['at'])
        
        oldest_date = df['at'].min().strftime('%Y-%m-%d %H:%M:%S')
        newest_date = df['at'].max().strftime('%Y-%m-%d %H:%M:%S')
        total_days = (df['at'].max() - df['at'].min()).days
        
        print(f" -> Total Data: {len(df)} baris")
        print(f" -> Tanggal Terlama (T0): {oldest_date}")
        print(f" -> Tanggal Terbaru (T1): {newest_date}")
        print(f" -> Rentang Waktu (Delta): {total_days} hari")
        
    except Exception as e:
        print(f"[ERROR] Inspeksi gagal: {e}")

if __name__ == "__main__":
    check_time_boundary('dataset_indosat/myim3_reviews_raw.csv')

[*] Menginspeksi rentang waktu: dataset_indosat/myim3_reviews_raw.csv
 -> Total Data: 11566 baris
 -> Tanggal Terlama (T0): 2026-04-20 00:15:06
 -> Tanggal Terbaru (T1): 2026-06-28 21:46:02
 -> Rentang Waktu (Delta): 69 hari
